## Executive Summary

**Purpose:** the leakage inductance of the flux that crosses the slot and the slot opening without reaching the air gap. It depends on **where each conductor sits in the slot**, so phases placed in different layers get different inductances, and phases sharing a slot are coupled.

**What it does:** `slot_permeance` gives the specific permeance matrix Λ between the named positions of a slot layout. Its parts are the conductor body (any taper from parallel slot to parallel tooth), the opening and tooth-tip transition, and the tooth-tip leakage at the gap. `slot_leakage_inductance` adds Λ over every slot with the turns of each phase at each position, giving the m × m matrix in the same form as `airgap_inductance`. The total stator inductance (without end winding) is the sum of the two, and `phase_imbalance` applies to it.

**Background:** the flux paths and how this part fits into the measured inductance are on the [Inductance Theory](inductance_theory.html) page.

**Assumptions:** iron of infinite permeability. The slot field is horizontal (across the slot) and uniform across its width. Turns are spread evenly over the area of their position. Eddy currents in the conductors are neglected (low frequency).

## How It Fits Into the Motor Design Workflow

**Upstream:** `WindingPattern` (turns of each phase at each slot position) and `StatorGeometry` (slot shape, its regions per layout, gap, stack).

**Downstream:** the total inductance $\mathbf{L} = \mathbf{L}_{ag} + \mathbf{L}_{slot} + \mathbf{L}_{ew}$, its imbalance, the synchronous and dq inductances, and comparing candidate windings and slot shapes.

## Theory

![Left: a slot with three radial layers, layer 1 next to the air gap and layer 3 at the slot bottom; horizontal leakage flux lines cross the slot at several depths and close through the teeth and yoke. A dashed line at depth y marks that flux crossing there links only the turns deeper than y. Right: the share of each layer's turns deeper than y, A_r(y), against depth, and the slot MMF F(y) when the three layers carry equal currents.](images/slot_leakage_field.svg)

### Step 1 — the field across the slot

Measure depth $y$ from the top of the conductor body (the air-gap side) down to the slot bottom at $y = h_s$. The body width is $b(y) = b_s + t\,2\pi y/Q$ (see [Stator and Slot Geometry](inductance_geometry.html)). Take a loop that crosses the slot at depth $y$ and closes through the two teeth and the yoke. With infinitely permeable iron all of the MMF is across the slot, so Ampère's law gives

$$H(y)\, b(y) = F(y) = \sum_r n_r\, i_r\, A_r(y),$$

with $n_r i_r$ the ampere-turns at slot position $r$ and $A_r(y)$ **the share of position $r$'s turns that lie deeper than $y$**, i.e. enclosed by the loop. $A_r = 1$ above the position, $0$ below it, and falls with the copper area in between.

### Step 2 — energy and the permeance matrix

The field energy per unit stack length is $W' = \tfrac12 \mu_0 \int H^2\, b\, dy = \tfrac12 \mu_0 \int F^2/b\, dy$. Writing $W' = \tfrac12 \sum_{rs} (n_r i_r)\,\mu_0 \Lambda_{rs}\,(n_s i_s)$ gives the **specific permeance matrix**

$$\boxed{\Lambda_{rs} = \int_0^{h_s} \frac{A_r(y)\,A_s(y)}{b(y)}\,dy \;+\; \lambda_{open} \;+\; \lambda_{tt}}$$

- **Body.** A conductor deep in the slot is enclosed by the flux crossing at every depth above it, so its self term $\Lambda_{rr}$ is large. A conductor next to the opening is enclosed by little. Two positions are coupled through the depths where both are enclosed: $\Lambda_{rs} = \int A_r A_s/b$.
- **Opening and tooth-tip transition** (heights $h_0$, $h_1$). Every conductor is deeper, $A_r = 1$, so they add the same constant to every entry: $\lambda_{open} = h_0/b_0 + h_1 \ln(b_s/b_0)/(b_s - b_0)$. The last term is $\int dy/b$ over a width that grows linearly from $b_0$ to $b_s$.
- **Tooth-tip leakage** across the opening at the gap: $\lambda_{tt} = \dfrac{5\,g/b_0}{5 + 4\,g/b_0}$ (Pyrhönen). It links every conductor too.

### Step 3 — phase inductance

The ampere-turns at position $r$ of slot $i$ are $\sum_x N_{x,i,r}\, i_x$, with $N$ the turns per ampere of each phase (`pattern.effective_turns()`). Summing the energy over the slots and stack:

$$L_{slot,xy} = \mu_0\, l_{stk} \sum_{i=0}^{Q-1} \sum_{r,s} N_{x,i,r}\, \Lambda_{rs}\, N_{y,i,s}.$$

Phases that never share a slot have no mutual slot leakage. Phases that share one are coupled through $\Lambda_{rs}$, with $r \ne s$.

### Worked case: three equal layers in a rectangular slot

For layer height $h$ and width $b$ ($A_r$ falls linearly across layer $r$), the body part in units of $h/b$ is

| | layer 1 (gap side) | layer 2 | layer 3 (bottom) |
|---|---|---|---|
| **layer 1** | 1/3 | 1/2 | 1/2 |
| **layer 2** | 1/2 | 4/3 | 3/2 |
| **layer 3** | 1/2 | 3/2 | 7/3 |

So the self slot leakage of the three layers is in the ratio **1 : 4 : 7**. A winding with one phase per layer is therefore unbalanced, however balanced its air-gap part is. The diagonal grows with depth because the flux crossing above a conductor adds up as it goes deeper. The single-layer limit, $\Lambda = h_s/(3 b_s)$, is the textbook slot permeance factor.

### Can the slot shape equalise the layers?

Only partly. Widening the slot towards the bottom (taper $t \to 1$, parallel tooth) lowers $1/b(y)$ where the deep layers collect their linkage, so their self permeance falls more than that of layer 1. A deeper, narrower opening and the tooth tip add the same constant to every layer, which dilutes the *relative* spread. But every layer's linkage is an integral of a positive quantity over the depths above it, so a deeper layer always has the larger self permeance: **no slot shape can make the self leakage of the layers equal**. The ways to balance it are in the winding: rotate the phases through the layers from pole to pole (transposition), or keep each phase in the same mix of layers.

In [ ]:
#| default_exp inductance.slot

In [ ]:
#| hide
import matplotlib.pyplot as plt
from dataclasses import replace
from emachines.inductance.geometry import SlotGeometry
from emachines.inductance.airgap import airgap_inductance, phase_imbalance
from emachines.winding.pattern import Coil

In [ ]:
#| export
from __future__ import annotations

import numpy as np

from emachines.inductance.airgap import MU0
from emachines.inductance.geometry import SlotRegion, StatorGeometry
from emachines.winding.pattern import WindingPattern

SLOT_PARTS = ("body", "opening", "slot", "tooth_tip", "total")
_GL_X, _GL_W = np.polynomial.legendre.leggauss(24)

In [ ]:
#| export
def _below_share(slot, Q: int, reg: SlotRegion, y: np.ndarray) -> np.ndarray:
    """Share of a region's turns that lie deeper than y (1 above the region, 0 below it)."""
    yc = np.clip(y, reg.y0, reg.y1)
    return (_area_to(slot, Q, reg.y1) - _area_to(slot, Q, yc)) / slot.area(Q, reg.y0, reg.y1)


def _area_to(slot, Q: int, y):
    """Body cross-section between the top of the body and depth y (vectorised)."""
    return slot.bs * y + slot.t * np.pi / Q * y ** 2


def body_permeance(stator: StatorGeometry, regions) -> np.ndarray:
    """
    Specific permeance matrix of the conductor body between slot positions (dimensionless).

    Λ_rs = ∫₀^hs A_r(y) A_s(y) / b(y) dy, with A_r(y) the share of position r's turns
    deeper than y and b(y) the body width. Integrated exactly (Gauss–Legendre per segment
    between region boundaries; the integrand is smooth inside each segment).

    Parameters
    ----------
    stator  : StatorGeometry
    regions : sequence of SlotRegion   e.g. ``stator.regions(layout, n)``

    Returns
    -------
    np.ndarray, shape (k, k)   symmetric, k = len(regions)
    """
    slot, Q = stator.slot, stator.Q
    cuts = np.unique(np.clip([0.0, slot.hs] + [v for r in regions for v in (r.y0, r.y1)], 0.0, slot.hs))
    lam = np.zeros((len(regions), len(regions)))
    for a, b in zip(cuts[:-1], cuts[1:]):
        y = 0.5 * (a + b) + 0.5 * (b - a) * _GL_X
        w = 0.5 * (b - a) * _GL_W / slot.width(y, Q)
        A = np.array([_below_share(slot, Q, r, y) for r in regions])
        lam += (A * w) @ A.T
    return lam


def opening_permeance(stator: StatorGeometry) -> float:
    """
    Specific permeance of the slot opening and tooth-tip transition above the conductor body.

    h0 / b0 + h1 · ln(bs / b0) / (bs − b0)  (the transition widens linearly from b0 to bs).
    Every conductor in the slot lies below it, so it links all of them fully. 0 for an open slot.
    """
    s = stator.slot
    lam = s.h0 / s.b0 if s.h0 > 0 else 0.0
    if s.h1 > 0:
        lam += s.h1 * np.log(s.bs / s.b0) / (s.bs - s.b0) if s.bs > s.b0 else s.h1 / s.bs
    return float(lam)


def tooth_tip_permeance(stator: StatorGeometry) -> float:
    """
    Specific permeance of the tooth-tip leakage across the slot opening at the air gap.

    λ_tt = 5 (g / b0) / (5 + 4 g / b0)  (Pyrhönen et al., §4.3, without the pitch factor:
    the pitch enters through the actual slot currents). Links every conductor in the slot.
    """
    r = stator.g / stator.slot.b0
    return float(5 * r / (5 + 4 * r))

In [ ]:
#| export
def slot_permeance(stator: StatorGeometry, layout: str, n: int = 2, part: str = "total") -> np.ndarray:
    """
    Specific permeance matrix between the named positions of a slot layout (dimensionless).

    The leakage flux linkage of the slot is μ0 l_stk nᵀ Λ n, with n the turns × current
    at each position. Diagonal entries are the self permeance of each position, off-diagonal
    entries the mutual permeance between positions.

    Parameters
    ----------
    stator : StatorGeometry
    layout : str   'single', 'over_under' or 'side_by_side'
    n      : int   Radial layers for 'over_under'
    part   : str   'body' (conductor region), 'opening' (h0 and h1 above the body),
                   'slot' (body + opening), 'tooth_tip', or 'total' (slot + tooth tip)

    Returns
    -------
    np.ndarray, shape (k, k), in the order of ``slot_positions(layout, n)``
    """
    if part not in SLOT_PARTS:
        raise ValueError(f"part must be one of {SLOT_PARTS}, got {part!r}")
    regions = stator.regions(layout, n)
    k = len(regions)
    ones = np.ones((k, k))
    terms = {"body": lambda: body_permeance(stator, regions),
             "opening": lambda: opening_permeance(stator) * ones,
             "tooth_tip": lambda: tooth_tip_permeance(stator) * ones}
    use = {"body": ["body"], "opening": ["opening"], "tooth_tip": ["tooth_tip"],
           "slot": ["body", "opening"], "total": ["body", "opening", "tooth_tip"]}[part]
    lam = np.zeros((k, k))
    for t in use:
        lam = lam + terms[t]()
    return lam


def slot_leakage_inductance(pattern: WindingPattern, stator: StatorGeometry, part: str = "total") -> np.ndarray:
    """
    Slot (and tooth-tip) leakage inductance matrix of the phases [H].

    L_xy = μ0 l_stk Σ_slots Σ_r Σ_s N_x,r Λ_rs N_y,s, with N the turns per ampere of
    each phase at each slot position (``pattern.effective_turns()``).

    Parameters
    ----------
    pattern : WindingPattern   Its Q must match the stator
    stator  : StatorGeometry
    part    : str              as ``slot_permeance``: 'body', 'opening', 'slot', 'tooth_tip' or 'total'

    Returns
    -------
    np.ndarray, shape (m, m)   symmetric
    """
    if pattern.Q != stator.Q:
        raise ValueError(f"the winding has Q={pattern.Q} slots but the stator has Q={stator.Q}")
    lam = slot_permeance(stator, pattern.layout, len(pattern.positions), part)
    order = [r.name for r in stator.regions(pattern.layout, len(pattern.positions))]
    idx = [list(pattern.positions).index(nm) for nm in order]
    N = pattern.effective_turns()[:, :, idx]            # (m, Q, k) in region order
    return MU0 * stator.l_stk * np.einsum("xir,rs,yis->xy", N, lam, N)

---

## Examples

The 36-slot stator from the geometry notebook (semi-closed slot, taper 0.5) with a short-pitched double layer. Its leakage is split into body, opening and tooth tip, and compared with the air-gap part:

In [ ]:
stator = StatorGeometry(Q=36, r_bore=60e-3, g=0.8e-3, l_stk=100e-3, h_y=12e-3, l_ew=22e-3,
                        slot=SlotGeometry.semi_closed(b0=2.5e-3, h0=1e-3, h1=1.5e-3, bs=6e-3, hs=18e-3, t=0.5))
pat = WindingPattern.distributed(36, 4, pitch=7, turns=10, parallel_paths=2)
print("Λ between top and bottom layer, total:\n", np.round(slot_permeance(stator, "over_under", 2), 4))
Ls = {}
for part in ("body", "opening", "tooth_tip", "total"):
    Ls[part] = phase_imbalance(slot_leakage_inductance(pat, stator, part))["L_positive"]
    print(f"{part:10s} synchronous {Ls[part] * 1e3:.4f} mH")
Lag = phase_imbalance(airgap_inductance(pat, stator))["L_positive"]
print(f"air gap    synchronous {Lag * 1e3:.4f} mH -> slot + tooth-tip leakage is {100 * Ls['total'] / Lag:.1f}% of it")

### A winding with one phase per layer (EP 2 237 392 B1)

The patent's 48-slot, 4-pole winding puts each phase in its own radial layer: 6 concentric coils per pole, phases A, B and C starting at slots 1, 5 and 9. Its air-gap part is balanced, but the slot leakage is not. The patent gives no dimensions, so the ones below are assumed: bore 80 mm, gap 0.6 mm, stack 150 mm, slot 5 × 24 mm. Phase A is taken as the layer next to the gap.

In [ ]:
def ep2237392(order=("1", "2", "3"), turns=2):
    # 48 slots, 4 poles, 3 layers; phase k uses layer order[k]; B's first pole reversed for 120° el.
    coils = []
    for ph, (start, sign0) in enumerate([(0, 1), (4, -1), (8, 1)]):
        for pole in range(4):
            s0, sign = start + 12 * pole, sign0 * (-1) ** pole
            for k in range(6):                                   # concentric: spans 1, 3, ..., 11
                a, b = (s0 + 5 - k) % 48, (s0 + 6 + k) % 48
                if sign < 0:
                    a, b = b, a
                coils.append(Coil(ph, a, order[ph], b, order[ph], turns))
    return WindingPattern.custom(48, 4, coils, layout="over_under", n_layers=3, name="EP2237392")

ep = ep2237392()
ep_open = StatorGeometry(Q=48, r_bore=80e-3, g=0.6e-3, l_stk=150e-3, h_y=15e-3, slot=SlotGeometry.open(5e-3, 24e-3))
ep_semi = replace(ep_open, slot=SlotGeometry.semi_closed(b0=2e-3, h0=1e-3, h1=1.5e-3, bs=5e-3, hs=24e-3))
print("air-gap balanced:", ep.check()["balanced"], " kw1 =", np.round(ep.check()["kw1"], 4))
for st, name in [(ep_open, "open slot"), (ep_semi, "semi-closed slot")]:
    Lag, Lsl = airgap_inductance(ep, st), slot_leakage_inductance(ep, st)
    for label, L in [("air gap", Lag), ("slot + tooth tip", Lsl), ("total", Lag + Lsl)]:
        r = phase_imbalance(L)
        print(f"{name:16s} {label:16s} self A/B/C [µH] {np.round(np.diag(L) * 1e6, 1)}  "
              f"self spread {100 * r['self_spread']:5.2f}%  sequence coupling {100 * r['sequence_coupling']:.2f}%")

The air-gap part is the same for every phase. The slot leakage of the layers is close to 1 : 4 : 7, plus the constant from the opening and tooth tip, and that sets the imbalance of the total. Rotating the layers from pole to pole (A in layer 1 under pole 1, layer 2 under pole 2, …) spreads each phase over all depths. With 4 poles and 3 layers it can't come out exactly equal, but it comes close:

In [ ]:
def ep_rotated(turns=2):
    coils = []
    for ph, (start, sign0) in enumerate([(0, 1), (4, -1), (8, 1)]):
        for pole in range(4):
            s0, sign = start + 12 * pole, sign0 * (-1) ** pole
            pos = str((ph + pole) % 3 + 1)                       # layer rotates with the pole
            for k in range(6):
                a, b = (s0 + 5 - k) % 48, (s0 + 6 + k) % 48
                if sign < 0:
                    a, b = b, a
                coils.append(Coil(ph, a, pos, b, pos, turns))
    return WindingPattern.custom(48, 4, coils, layout="over_under", n_layers=3, name="EP rotated")

for p_ in (ep, ep_rotated()):
    r = phase_imbalance(airgap_inductance(p_, ep_open) + slot_leakage_inductance(p_, ep_open))
    print(f"{p_.name:12s} total self spread {100 * r['self_spread']:.2f}%, sequence coupling {100 * r['sequence_coupling']:.3f}%")

How the slot shape changes the layer imbalance. The plot shows the self permeance of each layer and the relative spread $(\Lambda_{33} - \Lambda_{11})/\overline{\Lambda}$, as the taper goes from parallel slot to parallel tooth, for an open and a semi-closed slot:

![Left: self permeance of layers 1, 2 and 3 against taper t for the EP stator with an open slot; all fall as t grows, layer 3 most. Right: relative spread of the layer self permeance against t, for the open and the semi-closed slot; the taper lowers it a little, the semi-closed opening lowers it more, and neither brings it near zero.](images/layer_taper.svg)

In [ ]:
for t in (0.0, 0.5, 1.0):
    for st, name in [(ep_open, "open"), (ep_semi, "semi-closed")]:
        s = st.slot
        st_t = replace(st, slot=replace(s, t=t))
        d = np.diag(slot_permeance(st_t, "over_under", 3))
        print(f"t = {t:.1f} {name:12s} layer self permeance {np.round(d, 3)}  spread {100 * (d[2] - d[0]) / d.mean():.1f}%")

---

## Validation

1. **Three equal layers, rectangular slot:** the body matrix is exactly $\frac{h}{b}\begin{bmatrix}1/3 & 1/2 & 1/2\\ 1/2 & 4/3 & 3/2\\ 1/2 & 3/2 & 7/3\end{bmatrix}$, and one full-depth region gives $h_s/(3 b_s)$.
2. **Short pitch (Pyrhönen's $k_1$, $k_2$):** for a 60°-belt double layer with relative shortening $\varepsilon = 1 - W/\tau_p \le 1/3$, the synchronous slot leakage is the full-pitch value times $k_1 = 1 - \tfrac{9}{16}\varepsilon$ for the body and $k_2 = 1 - \tfrac34\varepsilon$ for the opening. The model reproduces both exactly, without using them.
3. **Tapered slot:** the Gauss–Legendre integral matches an independent adaptive quadrature of $\int (1 - a(y)/a_{tot})^2/b(y)\,dy$ for a single region.
4. **Energy is positive:** Λ is symmetric positive semi-definite for every layout.

In [ ]:
# 1. three layers and one region, rectangular
st3 = StatorGeometry(Q=48, r_bore=80e-3, g=0.6e-3, l_stk=0.15, h_y=15e-3, slot=SlotGeometry.open(5e-3, 24e-3))
ref = np.array([[1/3, 1/2, 1/2], [1/2, 4/3, 3/2], [1/2, 3/2, 7/3]]) * (8e-3 / 5e-3)
assert np.allclose(slot_permeance(st3, "over_under", 3, "body"), ref, rtol=1e-12)
assert np.isclose(slot_permeance(st3, "single", part="body")[0, 0], 24e-3 / (3 * 5e-3), rtol=1e-12)
print("3-layer matrix [h/b]:\n", np.round(slot_permeance(st3, "over_under", 3, "body") / (8e-3 / 5e-3), 6))

# 2. short pitch factors k1 (body) and k2 (opening)
full = WindingPattern.distributed(36, 4, pitch=9)
for pitch in (8, 7, 6):
    eps = 1 - pitch / 9
    p_ = WindingPattern.distributed(36, 4, pitch=pitch)
    k = {part: phase_imbalance(slot_leakage_inductance(p_, stator, part))["L_positive"]
         / phase_imbalance(slot_leakage_inductance(full, stator, part))["L_positive"] for part in ("body", "opening")}
    print(f"pitch {pitch}/9: k1 = {k['body']:.6f} (expected {1 - 9 * eps / 16:.6f} for t = 0 only), "
          f"k2 = {k['opening']:.6f} (expected {1 - 3 * eps / 4:.6f})")
    assert np.isclose(k["opening"], 1 - 3 * eps / 4)
    st0 = replace(stator, slot=replace(stator.slot, t=0.0))
    k1 = (phase_imbalance(slot_leakage_inductance(p_, st0, "body"))["L_positive"]
          / phase_imbalance(slot_leakage_inductance(full, st0, "body"))["L_positive"])
    assert np.isclose(k1, 1 - 9 * eps / 16)

$k_1$ is exact for a parallel slot ($t = 0$). With a taper, the body share depends slightly on the pitch through $b(y)$; the model handles that, and the textbook factor does not.

---

## Tests

In [ ]:
#| hide
from scipy.integrate import quad

# 3. taper: one region against adaptive quadrature
stt = replace(st3, slot=SlotGeometry.open(5e-3, 24e-3, t=0.7))
s = stt.slot
ref = quad(lambda y: (1 - s.area(48, 0, y) / s.area(48)) ** 2 / float(s.width(y, 48)), 0, s.hs, epsabs=0, epsrel=1e-12)[0]
assert np.isclose(slot_permeance(stt, "single", part="body")[0, 0], ref, rtol=1e-10)
# ... and a layer boundary inside the taper: the full-slot sum is the single-region value
assert np.isclose(slot_permeance(stt, "over_under", 3, "body").sum(), 9 * ref, rtol=1e-10)

# 4. symmetric positive semi-definite for every layout; side-by-side halves are fully coupled (1-D field)
for lay, n in [("single", 1), ("over_under", 2), ("over_under", 4), ("side_by_side", 2)]:
    lam = slot_permeance(stator, lay, n)
    assert np.allclose(lam, lam.T) and np.linalg.eigvalsh(lam).min() > -1e-12
assert np.allclose(slot_permeance(stator, "side_by_side", 2), slot_permeance(stator, "single")[0, 0])

# open slot: no opening term; tooth tip uses bs
assert slot_permeance(st3, "over_under", 3, "opening").max() == 0
assert np.isclose(tooth_tip_permeance(st3), 5 * (0.6 / 5) / (5 + 4 * 0.6 / 5))
# parts add up; balanced windings stay balanced
assert np.allclose(slot_permeance(stator, "over_under", 2, "total"),
                   sum(slot_permeance(stator, "over_under", 2, p) for p in ("body", "opening", "tooth_tip")))
for p_ in [WindingPattern.distributed(36, 4, pitch=7), WindingPattern.distributed(36, 4, layers=1)]:
    r = phase_imbalance(slot_leakage_inductance(p_, stator))
    assert r["self_spread"] < 1e-12 and r["sequence_coupling"] < 1e-12
for p_ in [WindingPattern.concentrated(12, 10), WindingPattern.concentrated(12, 10, layout="over_under")]:
    st12 = StatorGeometry(Q=12, r_bore=50e-3, g=0.7e-3, l_stk=80e-3, h_y=10e-3, slot=SlotGeometry.open(8e-3, 20e-3))
    r = phase_imbalance(slot_leakage_inductance(p_, st12))
    assert r["self_spread"] < 1e-12 and r["sequence_coupling"] < 1e-12
# scaling with turns, parallel paths and stack length
L1 = slot_leakage_inductance(WindingPattern.distributed(36, 4, pitch=7), stator)
assert np.allclose(slot_leakage_inductance(WindingPattern.distributed(36, 4, pitch=7, turns=6, parallel_paths=3), stator), 4 * L1)
assert np.allclose(slot_leakage_inductance(WindingPattern.distributed(36, 4, pitch=7), replace(stator, l_stk=0.2)), 2 * L1)
# the EP winding: layer ratio 1:4:7 on the body part with one phase per layer
d = np.diag(slot_leakage_inductance(ep, ep_open, "body"))
assert np.allclose(d / d[0], [1, 4, 7])
# errors
for f in [lambda: slot_permeance(stator, "over_under", 2, "end"),
          lambda: slot_leakage_inductance(WindingPattern.distributed(24, 4), stator)]:
    try:
        f(); raise AssertionError("expected ValueError")
    except ValueError:
        pass

## References

- Pyrhönen, J., Jokinen, T., Hrabovcová, V. (2014). *Design of Rotating Electrical Machines* (2nd ed.). Wiley, §4.3 — slot leakage permeance factors, short-pitch factors $k_1$, $k_2$, tooth-tip leakage.
- Sudhoff, S.D. *Power Magnetic Devices* (2nd ed.), §4.7 and chapter 8 — slot leakage by energy, multi-conductor slots.
- Lipo, T.A. (2017). *Introduction to AC Machine Design*. Wiley-IEEE Press, chapter 6 — slot leakage of multi-layer windings, mutual slot leakage between phases.
- EP 2 237 392 B1, *AC motor winding pattern* (granted 2018) — the one-phase-per-layer winding used in the example (dimensions assumed).